# Baseline_v24 — Qwen3.5-2B

v14 的序列选择与相邻切片三通道构造；视觉全量微调、语言模型 LoRA、12 标签分类头。

挂载 best.pt、完整 Qwen 基础模型及所需离线 wheels。首次运行先修改下面的路径。互联网保持关闭；推理沿用 checkpoint 的窗口数、裁剪、元数据和训练覆盖阈值。


In [ ]:
from pathlib import Path
import os

# Set these exact paths to your attached Kaggle datasets/model snapshot.
DATA_ROOT = Path("/kaggle/input/rsna-knee-abnormality-detection")
CHECKPOINT_PATH = Path("/kaggle/input/rsna-v24-weights/best.pt")
QWEN_MODEL_DIR = None  # Auto-detect the attached Qwen3.5-2B snapshot; set a Path to override.
WHEEL_DIR = None  # e.g. "/kaggle/input/rsna-v24-wheels"; install before importing Transformers.
USE_ALL_GPUS = True
BATCH_SIZE = 1
NUM_WORKERS = 2
AMP = "auto"  # bf16 when supported; otherwise fp16.
ATTENTION = "sdpa"
SDPA_BACKEND = "math"  # Set auto to permit fused PyTorch kernels after testing the target device.
EVAL_WINDOWS = None  # Follow checkpoint; 0 uses all v14 candidates, at greater cost.
OUTPUT = Path("/kaggle/working/submission.csv")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
import importlib.metadata
import subprocess
import sys
import json

if QWEN_MODEL_DIR is None:
    candidates = []
    for config_path in Path("/kaggle/input").rglob("config.json"):
        try:
            config = json.loads(config_path.read_text(encoding="utf-8"))
        except (ValueError, OSError):
            continue
        text = config.get("text_config", {})
        if (config.get("model_type") == "qwen3_5" and config.get("vision_config")
                and text.get("hidden_size") == 2048 and text.get("num_hidden_layers") == 24):
            candidates.append(config_path.parent)
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one attached Qwen3.5-2B snapshot, found {candidates}; set QWEN_MODEL_DIR explicitly.")
    QWEN_MODEL_DIR = candidates[0]
else:
    QWEN_MODEL_DIR = Path(QWEN_MODEL_DIR)
print("Qwen model:", QWEN_MODEL_DIR)

required = {"transformers": "5.18.0", "peft": "0.21.2", "accelerate": "1.15.0"}
if WHEEL_DIR is not None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", str(WHEEL_DIR),
                           *[f"{name}=={version}" for name, version in required.items()],
                           "pydicom>=3.0", "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"])
for name, expected in required.items():
    try:
        actual = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError as exc:
        raise RuntimeError(f"Missing {name}; attach offline wheels and set WHEEL_DIR before running.") from exc
    if actual != expected:
        raise RuntimeError(f"{name}={actual}, expected {expected}; use the supplied offline wheel workflow.")
import torch
from packaging.version import Version
if Version(torch.__version__.split("+")[0]) < Version("2.9"):
    raise RuntimeError("This v24 environment requires torch>=2.9 and matching torchvision; prepare compatible offline wheels.")
print("torch:", torch.__version__, "GPUs:", torch.cuda.device_count())
for path in (DATA_ROOT / "test.csv", DATA_ROOT / "test_series.csv", CHECKPOINT_PATH, QWEN_MODEL_DIR / "config.json"):
    if not path.is_file():
        raise FileNotFoundError(path)


In [ ]:
import json
from pathlib import Path

SOURCES = {'rsna_data.py': '# -*- coding: utf-8 -*-\n"""RSNA Knee 数据加载与 140 mm physical crop 预处理模块。"""\n\nimport hashlib\nimport json\nimport os\nimport uuid\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport pydicom\nfrom pydicom.errors import InvalidDicomError\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\n\nDEFAULT_ROOT = Path("/root/RSNA/rsna-knee-abnormality-detection")\nDEFAULT_IMAGE_SIZE = 336\nDEFAULT_CROP_MM = 140.0\n# 一个 Study 预测 12 个标签；五个 slot 按方位/Fluid Sensitive 偏好选不同序列。\nLABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nSLOTS = ["Sagittal_fluid", "Sagittal_second", "Coronal_fluid", "Coronal_second", "Axial"]\nSLOT_SPECS = [("Sagittal", 1, 23), ("Sagittal", 0, 17), ("Coronal", 1, 15), ("Coronal", 0, 10), ("Axial", None, 15)]\nCANDIDATE_BUDGETS = [spec[2] for spec in SLOT_SPECS]\nCACHE_SCHEMA = 1\nSERIES_SELECTION_VERSION = "header_quality_v1"\nQUALITY_COLUMNS = ["quality_usable", "quality_header_errors", "quality_geometry_errors",\n                   "quality_unique_slices", "quality_coverage_mm", "quality_pixel_spacing_mm",\n                   "quality_slice_spacing_mm", "quality_fov_mm"]\n# 每个序列对应 11 个数值元数据，顺序必须与 load_series 构造 features 的顺序一致。\nSERIES_FEATURES = ["fluid_sensitive", "fat_suppression", "t1", "pd", "t2", "contrast_unknown", "te", "tr", "pixel_spacing", "slice_spacing", "coverage"] #翻译：\n#MRI 序列特征：      液体敏感；         脂肪抑制； T1 加权；质子密度加权；T2 加权；对比或加权类型未知；回波时间；重复时间；像素间距；相邻切片间距；扫描覆盖范围。\n\ndef _count_dicoms(path):\n    return sum(1 for _ in Path(path).glob("*.dcm"))\n\n\ndef _prepare_series_df(series_df, image_root):\n    # 在原始序列表上补路径、槽位和切片数，方便后续按 Study/槽位选序列。\n    series_df = series_df.copy()\n    series_df["SeriesPath"] = [Path(image_root) / str(study) / str(series) for study, series in zip(series_df.StudyInstanceUID, series_df.SeriesInstanceUID)]\n    series_df["NumSlices"] = [_count_dicoms(path) for path in series_df.SeriesPath]\n    return series_df\n\n\ndef load_metadata(root=DEFAULT_ROOT):\n    # Study 标签表与序列描述表分别读取；实际 DICOM 只在 __getitem__ 时按需解码。\n    root = Path(root)\n    train_df, test_df = pd.read_csv(root / "train.csv"), pd.read_csv(root / "test.csv")\n    train_series_df = _prepare_series_df(pd.read_csv(root / "train_series.csv"), root / "train_series")\n    test_series_df = _prepare_series_df(pd.read_csv(root / "test_series.csv"), root / "test_series")\n    return train_df, train_series_df, test_df, test_series_df, pd.read_csv(root / "sample_submission.csv")\n\n\ndef get_sorted_dicom_info(series_path):\n\n    """使用患者坐标排序，并把主切片方向统一为患者坐标轴的正方向。"""\n\n    # 获取当前MRI序列文件夹下所有DICOM文件路径\n    files = list(Path(series_path).glob("*.dcm"))\n    if not files:\n        raise FileNotFoundError(f"No DICOM files found in series: {series_path}")\n    # 读取所有DICOM文件的头信息，但不读取像素数据，节省内存\n    headers = [pydicom.dcmread(file, stop_before_pixels=True) for file in files]\n    # 取第一张切片的ImageOrientationPatient，表示该图片在患者空间中的行方向和列方向   #ImagePositionPatient表示这一张图片左上角在患者三维空间的位置\n    orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=np.float32)\n    # 前3个数表示图片行方向，后3个数表示图片列方向，两个方向叉乘得到切片法向量\n    normal = np.cross(orientation[:3], orientation[3:])\n    # 将法向量归一化，使其长度变为1，方便后续进行空间投影计算\n    normal /= np.linalg.norm(normal)\n    # 判断法向量主要方向是否为负，如果是则翻转，保证所有序列方向统一\n    if normal[np.argmax(np.abs(normal))] < 0:\n        # 反转法向量方向\n        normal = -normal\n    # 将每张切片的患者空间坐标投影到切片法向量方向，得到每张slice的真实空间位置\n    positions = np.asarray([np.dot(np.asarray(header.ImagePositionPatient, dtype=np.float32), normal) for header in headers])\n    # 根据真实空间位置从小到大排序，得到正确的slice顺序\n    order = np.argsort(positions)\n    # 按照排序后的索引，同时返回：\n    # 1. 排序后的DICOM文件 2. 对应空间位置 3. 对应header信息\n    return [files[i] for i in order], positions[order], [headers[i] for i in order]\n\n\ndef sort_dicom_files(series_path):\n    # 只需要已排序文件列表时的便捷包装。\n    return get_sorted_dicom_info(series_path)[0]\n\n#读取一张DICOM MRI切片，将其转换成标准的二维numpy图像矩阵，并处理少数异常DICOM文件和灰度方向问题。\ndef read_dicom(path):\n    dicom = pydicom.dcmread(path)\n    # 尝试直接通过pydicom解析PixelData得到图像矩阵\n    try:\n        # pixel_array会自动根据DICOM头信息解析像素，并转换成numpy数组\n        image = dicom.pixel_array.astype(np.float32)\n    # 如果标准解析失败，进入手动恢复流程\n    except ValueError as error:\n        # 获取图像高度和宽度，后续用于手动reshape像素数据\n        rows, columns = int(dicom.Rows), int(dicom.Columns)\n        # 获取帧数，多帧DICOM没有该字段时默认认为只有1帧\n        frames = int(getattr(dicom, "NumberOfFrames", 1) or 1)\n        # 获取每个像素包含几个采样值，例如RGB可能是3，MRI通常是1\n        samples = int(getattr(dicom, "SamplesPerPixel", 1) or 1)\n        # 计算理论上的像素总数量\n        pixel_count = rows * columns * frames * samples\n        # 直接读取DICOM中的原始像素字节数据\n        raw = dicom.PixelData\n        # 获取DICOM传输格式，用于判断PixelData是否经过压缩\n        transfer_syntax = getattr(getattr(dicom, "file_meta", None), "TransferSyntaxUID", None)\n        # 判断当前DICOM是否为未压缩格式\n        uncompressed = transfer_syntax is None or not transfer_syntax.is_compressed\n\n        # 只有满足以下条件才允许手动解析：\n        # 1. 未压缩\n        # 2. 单帧\n        # 3. 单通道\n        # 4. 原始字节数量符合预期\n        if not uncompressed or frames != 1 or samples != 1 or len(raw) not in (pixel_count, pixel_count + 1):\n\n            # 如果情况复杂，无法安全恢复，则直接抛出异常\n            raise ValueError(f"Cannot decode DICOM {path}: {error}") from error\n\n        # 根据PixelRepresentation判断像素是否为有符号整数\n        # 0代表unsigned，1代表signed\n        dtype = np.int8 if int(getattr(dicom, "PixelRepresentation", 0) or 0) else np.uint8\n\n        # 从原始字节创建numpy数组，并恢复成二维图像\n        image = np.frombuffer(raw[:pixel_count], dtype=dtype).reshape(rows, columns).astype(np.float32)\n\n    # 根据DICOM标准应用灰度线性变换\n    # 输出值 = 原始像素值 × RescaleSlope + RescaleIntercept\n    image = image * float(getattr(dicom, "RescaleSlope", 1)) + float(getattr(dicom, "RescaleIntercept", 0))\n\n    # MONOCHROME1表示像素越大越黑，需要反转成普通医学图像亮度方向\n    if getattr(dicom, "PhotometricInterpretation", "") == "MONOCHROME1":\n        # 使用最大值+最小值减去原像素，实现灰度反转\n        image = image.max() + image.min() - image\n\n    # 返回最终处理好的二维MRI切片\n    return image\n\n#读取指定的MRI切片，并且在遇到损坏DICOM时，用距离最近的正常切片替代，最后把所有slice组合成3D volume。注意这里的volume是一个numpy数组，shape为[S,H,W]，其中S是切片数，H和W是每张切片的高度和宽度。不是channle为3\ndef _read_sampled_slices(files, indices):\n    """读取采样切片；单张损坏时用空间上最近的可读切片替代。"""\n    # cache 避免重复解码同一张；invalid 避免对已知坏文件反复尝试。\n    cache, invalid = {}, set()\n    images = []\n    for index in indices:\n        index = int(index)\n        # 按切片索引距离从近到远尝试，直到找到可解码的替代切片。\n        candidates = sorted(range(len(files)), key=lambda candidate: abs(candidate - index))\n        for candidate in candidates:\n            if candidate in cache:\n                image = cache[candidate]\n                break\n            if candidate in invalid:\n                continue\n            try:\n                image = read_dicom(files[candidate])\n                cache[candidate] = image\n                break\n            except (ValueError, OSError, EOFError, InvalidDicomError):\n                invalid.add(candidate)\n        else:\n            raise RuntimeError(f"No decodable DICOM slices in series: {Path(files[0]).parent}")\n        images.append(image)\n    return np.stack(images)\n\n\ndef _sample_by_position(positions, num_slices):\n    """在真实物理覆盖范围内均匀采样，而不是只按切片编号采样。"""\n    # targets 是等间隔物理坐标；每个目标位置选最近的真实 DICOM 切片。\n    targets = np.linspace(positions[0], positions[-1], num_slices)\n    return np.abs(positions[:, None] - targets[None, :]).argmin(axis=0)\n\n\n#从一个MRI序列的DICOM header中读取像素空间大小（PixelSpacing），用于后续把“毫米尺度的crop”转换成“像素尺度的crop”。\ndef _get_pixel_spacing(headers, series_path):\n\n    """读取有效的行、列 PixelSpacing；优先使用堆栈中央切片。"""\n\n    # 计算DICOM序列中间位置的slice索引，优先检查中间切片\n    middle = len(headers) // 2\n    # 按距离中间切片的远近排序，生成检查顺序\n    search_order = sorted(range(len(headers)), key=lambda index: abs(index - middle))\n    # 按照从中间到两边的顺序遍历每一张slice的header\n    for index in search_order:\n\n        # 获取当前slice的PixelSpacing字段，如果不存在则返回None\n        value = getattr(headers[index], "PixelSpacing", None)\n\n        # 如果当前slice没有PixelSpacing信息，跳过这一张slice\n        if value is None or len(value) < 2:\n            continue\n\n        # 将PixelSpacing转换成numpy数组，只保留行方向和列方向的两个spacing值\n        spacing = np.asarray(value[:2], dtype=np.float32)\n\n        # 检查spacing是否有效：1. 所有值不是NaN/inf 2. 所有值大于0\n        if np.all(np.isfinite(spacing)) and np.all(spacing > 0):\n            # 返回有效的像素间距，单位通常是mm/pixel\n            return spacing\n    # 如果遍历所有slice都没有找到有效PixelSpacing，则说明该序列无法确定物理尺寸\n    raise ValueError(f"Series has no valid PixelSpacing: {series_path}")\n\n\ndef physical_center_crop(volume, pixel_spacing, crop_mm=DEFAULT_CROP_MM):\n\n    """按真实毫米尺度中心裁剪；原始 FOV 不足时先对称补零以保持尺度。"""\n\n    # 检查输入volume是否为三维MRI数据，格式应该是 [slice数量, height, width]\n    if volume.ndim != 3:\n        # 如果不是三维数据，说明输入格式错误，直接报错\n        raise ValueError(f"Expected volume [S,H,W], got shape {volume.shape}")\n    # 检查裁剪尺寸是否有效，crop_mm必须是正数\n    if not np.isfinite(crop_mm) or crop_mm <= 0:\n        # 如果crop_mm为负数、0或者无穷/NaN，则无法进行裁剪\n        raise ValueError(f"crop_mm must be positive, got {crop_mm}")\n    # 将pixel_spacing转换成numpy数组，方便后续数学计算\n    spacing = np.asarray(pixel_spacing, dtype=np.float32)\n\n    # 检查PixelSpacing是否合法：必须包含两个值，并且不能为NaN、inf，同时两个方向都必须大于0\n    if spacing.shape != (2,) or not np.all(np.isfinite(spacing)) or np.any(spacing <= 0):\n        # PixelSpacing异常时无法从毫米转换到像素，因此停止处理\n        raise ValueError(f"Invalid PixelSpacing: {pixel_spacing}")\n    # 根据真实毫米大小计算需要裁剪的像素高度 # 公式：目标像素数量 = crop毫米数 / 每个像素代表的毫米数\n    target_height = max(1, int(round(crop_mm / float(spacing[0]))))\n    # 根据真实毫米大小计算需要裁剪的像素宽度\n    target_width = max(1, int(round(crop_mm / float(spacing[1]))))\n    # 获取当前MRI volume的空间尺寸\n    # volume形状：[slice,height,width]\n    height, width = volume.shape[-2:]\n\n    # 计算高度方向需要补多少像素 # 如果原图高度已经大于目标高度，则不需要补\n    pad_height = max(0, target_height - height)\n    # 计算宽度方向需要补多少像素 # 如果原图宽度已经大于目标宽度，则不需要补\n    pad_width = max(0, target_width - width)\n\n    # 如果原始视野(FOV)小于目标裁剪区域，则先进行padding\n    if pad_height or pad_width:\n\n        # 原图不足140mm时进行左右/上下对称补零\n        # 避免直接放大较小视野导致空间尺度错误\n        volume = np.pad(\n            volume,\n\n            # 三个维度分别对应：\n            # slice维度不补，height方向补，width方向补\n            (\n                (0, 0),\n                # height方向上下对称补零\n                (pad_height // 2, pad_height - pad_height // 2),\n                # width方向左右对称补零\n                (pad_width // 2, pad_width - pad_width // 2),\n            ),\n\n            # 使用0填充，代表MRI背景区域\n            mode="constant",\n            # 指定填充值为0\n            constant_values=0,\n        )\n        # padding之后重新获取新的height和width\n        height, width = volume.shape[-2:]\n\n    # 计算中心裁剪开始位置的高度坐标\n    top = (height - target_height) // 2\n\n    # 计算中心裁剪开始位置的宽度坐标\n    left = (width - target_width) // 2\n\n    # 对所有slice使用同一个中心区域进行裁剪\n    # 保证整个3D volume的空间位置保持一致\n    return volume[:, top:top + target_height, left:left + target_width]\n\n\ndef _contrast_features(te, tr, fat_suppression):\n    """依据 EDA 使用的 TE/TR 启发式生成 T1、PD、T2、未知四维标记。"""\n    # 这些是经验规则推断的扫描对比度类别，不是人工标注的真值。\n    if fat_suppression:\n        return [0, 1, 1, 0]\n    if te <= 0 or tr <= 0:\n        return [0, 0, 0, 1]\n    if tr < 1000 and te < 30:\n        return [1, 0, 0, 0]\n    if te < 60:\n        return [0, 1, 0, 0]\n    return [0, 0, 1, 0]\n\n\ndef _binary_flag(value):\n    value = pd.to_numeric(value, errors="coerce")\n    return int(value) if pd.notna(value) and value in (0, 1) else None\n\n\ndef read_series_quality(series_path):\n    """只读 header 估计几何质量；不解码像素，不把 header 完整性当像素损坏率。"""\n    files = sorted(Path(series_path).glob("*.dcm"))\n    quality = dict.fromkeys(QUALITY_COLUMNS, None)\n    quality.update(quality_usable=0, quality_header_errors=0, quality_geometry_errors=0,\n                   quality_unique_slices=0)\n    headers, normals, locations = [], [], []\n    for file in files:\n        try:\n            header = pydicom.dcmread(file, stop_before_pixels=True)\n        except (OSError, ValueError, EOFError, InvalidDicomError):\n            quality["quality_header_errors"] += 1\n            continue\n        try:\n            orientation = np.asarray(header.ImageOrientationPatient, dtype=np.float64)\n            location = np.asarray(header.ImagePositionPatient, dtype=np.float64)\n            if orientation.shape != (6,) or location.shape != (3,):\n                raise ValueError("Invalid geometry dimensions")\n            if not np.isfinite(orientation).all() or not np.isfinite(location).all():\n                raise ValueError("Nonfinite geometry")\n            normal = np.cross(orientation[:3], orientation[3:])\n            length = float(np.linalg.norm(normal))\n            if length < 1e-6:\n                raise ValueError("Degenerate orientation")\n            normals.append(normal / length)\n            locations.append(location)\n            headers.append(header)\n        except (AttributeError, TypeError, ValueError):\n            quality["quality_geometry_errors"] += 1\n    if not headers:\n        return quality\n    normal = normals[0]\n    positions = np.asarray(locations) @ normal\n    # 合并相同位置的重建/重复切片，不以文件数量奖励重复图像。\n    unique = np.unique(np.round(positions, 4))\n    quality["quality_unique_slices"] = len(unique)\n    quality["quality_coverage_mm"] = float(unique[-1] - unique[0])\n    if len(unique) > 1:\n        quality["quality_slice_spacing_mm"] = float(np.median(np.diff(unique)))\n    ordered_headers = [headers[i] for i in np.argsort(positions)]\n    try:\n        spacing = _get_pixel_spacing(ordered_headers, series_path)\n        quality["quality_pixel_spacing_mm"] = float(spacing.max())\n        middle = ordered_headers[len(ordered_headers) // 2]\n        rows, columns = int(middle.Rows), int(middle.Columns)\n        if rows <= 0 or columns <= 0:\n            raise ValueError("Invalid image matrix")\n        quality["quality_fov_mm"] = float(min(rows * spacing[0], columns * spacing[1]))\n    except (AttributeError, TypeError, ValueError):\n        quality["quality_geometry_errors"] += 1\n    aligned = bool(np.all(np.abs(np.asarray(normals) @ normal) >= 0.999))\n    quality["quality_usable"] = int(\n        len(unique) >= 2 and aligned and quality["quality_header_errors"] == 0 and\n        quality["quality_geometry_errors"] == 0 and quality["quality_fov_mm"] is not None\n    )\n    return quality\n\n\ndef _cached_series_quality(series_path, cache_dir):\n    directory = Path(series_path)\n    cache_path = None\n    if cache_dir is not None:\n        files = []\n        for file in sorted(directory.glob("*.dcm")):\n            stat = file.stat()\n            files.append((file.name, stat.st_size, stat.st_mtime_ns))\n        signature = json.dumps([SERIES_SELECTION_VERSION, str(directory.resolve()), files])\n        cache_path = Path(cache_dir) / (hashlib.sha256(signature.encode()).hexdigest() + ".json")\n        if cache_path.is_file():\n            try:\n                cached = json.loads(cache_path.read_text(encoding="utf-8"))\n                if (isinstance(cached, dict) and cached.get("version") == SERIES_SELECTION_VERSION and\n                    isinstance(cached.get("quality"), dict) and all(k in cached["quality"] for k in QUALITY_COLUMNS)):\n                    return cached["quality"]\n            except (OSError, ValueError, KeyError, TypeError):\n                pass\n    quality = read_series_quality(directory)\n    if cache_path is not None:\n        cache_path.parent.mkdir(parents=True, exist_ok=True)\n        temporary = cache_path.with_name(f"{cache_path.name}.{os.getpid()}.{uuid.uuid4().hex}.tmp")\n        try:\n            temporary.write_text(json.dumps(dict(version=SERIES_SELECTION_VERSION, quality=quality), allow_nan=False),\n                                 encoding="utf-8")\n            os.replace(temporary, cache_path)\n        finally:\n            temporary.unlink(missing_ok=True)\n    return quality\n\n\ndef add_series_quality(series_df, cache_dir=None, workers=8):\n    """训练由 rank 0 扫描/读取缓存后广播；推理只扫描当前 GPU 分配的序列。"""\n    if workers < 1:\n        raise ValueError("Quality workers must be positive")\n    rows = series_df.copy()\n    print(f"Series quality: indexing {len(rows)} series (header-only, workers={workers})", flush=True)\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        results = []\n        for index, quality in enumerate(pool.map(lambda p: _cached_series_quality(p, cache_dir), rows.SeriesPath), 1):\n            results.append(quality)\n            if index % 1000 == 0 or index == len(rows):\n                print(f"Series quality: {index}/{len(rows)}", flush=True)\n    for column in QUALITY_COLUMNS:\n        rows[column] = [quality[column] for quality in results]\n    return rows\n\n\ndef make_series_selection_config(train_series_df, coverage_quantile=0.2):\n    """只使用弱标签训练 Study 的有效几何统计，确定各平面的覆盖参考阈值。"""\n    if not 0 <= coverage_quantile <= 1:\n        raise ValueError("Coverage quantile must be in [0,1]")\n    thresholds = {}\n    for plane in ("Sagittal", "Coronal", "Axial"):\n        values = train_series_df.loc[\n            train_series_df.Anatomical_Plane.eq(plane) & train_series_df.quality_usable.eq(1),\n            "quality_coverage_mm",\n        ]\n        values = pd.to_numeric(values, errors="coerce")\n        values = values[np.isfinite(values) & (values > 0)]\n        thresholds[plane] = float(values.quantile(coverage_quantile)) if len(values) else 0.0\n    config = dict(version=SERIES_SELECTION_VERSION, coverage_quantile=float(coverage_quantile),\n                  coverage_thresholds=thresholds)\n    validate_series_selection_config(config)\n    return config\n\n\ndef validate_series_selection_config(config):\n    if not isinstance(config, dict) or config.get("version") != SERIES_SELECTION_VERSION:\n        raise ValueError("v14 requires its checkpoint\'s header-quality series selection config")\n    quantile = config.get("coverage_quantile")\n    if not isinstance(quantile, (int, float)) or not np.isfinite(quantile) or not 0 <= quantile <= 1:\n        raise ValueError("Invalid coverage quantile")\n    thresholds = config.get("coverage_thresholds", {})\n    if not isinstance(thresholds, dict):\n        raise ValueError("Invalid coverage thresholds")\n    for plane in ("Sagittal", "Coronal", "Axial"):\n        value = thresholds.get(plane)\n        if not isinstance(value, (int, float)) or not np.isfinite(value) or value < 0:\n            raise ValueError(f"Invalid coverage threshold for {plane}")\n\n\ndef _series_quality_rank(record, config, crop_mm, image_size):\n    threshold = config["coverage_thresholds"][record.Anatomical_Plane]\n    coverage = float(record.quality_coverage_mm)\n    # 覆盖/FOV 达标后不再奖励更大范围，避免优先选大视野而低分辨率的序列。\n    coverage_score = min(coverage / threshold, 1.0) if threshold > 0 else 1.0\n    fov_score = min(float(record.quality_fov_mm) / crop_mm, 1.0)\n    effective_spacing = max(float(record.quality_pixel_spacing_mm), crop_mm / image_size)\n    return (coverage_score, fov_score, -effective_spacing,\n            -float(record.quality_slice_spacing_mm), int(record.quality_unique_slices))\n\n\ndef select_slot_series(rows, series_selection=None, crop_mm=DEFAULT_CROP_MM, image_size=DEFAULT_IMAGE_SIZE):\n    """v10 候选范围不变；配置存在时仅在同一集合内按 header 几何质量排序。"""\n    if series_selection is not None:\n        validate_series_selection_config(series_selection)\n        if any(column not in rows for column in QUALITY_COLUMNS):\n            raise ValueError("Call add_series_quality before v14 series selection")\n    used, selected = set(), []\n    for plane, preference, _ in SLOT_SPECS:\n        candidates = rows[(rows.Anatomical_Plane == plane) & (rows.NumSlices > 0)]\n        candidates = candidates[~candidates.SeriesInstanceUID.astype(str).isin(used)]\n        if preference is not None:\n            preferred = candidates[candidates.Fluid_Sensitive.map(_binary_flag).eq(preference)]\n            if len(preferred):\n                candidates = preferred\n        record = candidates.iloc[0] if len(candidates) else None\n        if series_selection is not None and len(candidates) > 1:\n            usable = candidates[candidates.quality_usable.eq(1)]\n            if len(usable):\n                # 同分保留首个元素；全部不可用时也回退 v10 CSV 顺序。\n                record = max((row for _, row in usable.iterrows()),\n                             key=lambda row: _series_quality_rank(row, series_selection, crop_mm, image_size))\n        selected.append(record)\n        if record is not None:\n            used.add(str(record.SeriesInstanceUID))\n    return selected\n\n\ndef series_selection_report(series_df, series_selection, crop_mm=DEFAULT_CROP_MM, image_size=DEFAULT_IMAGE_SIZE):\n    records = []\n    for uid, rows in series_df.groupby("StudyInstanceUID", sort=False):\n        original = select_slot_series(rows)\n        selected = select_slot_series(rows, series_selection, crop_mm, image_size)\n        for slot, old, new in zip(SLOTS, original, selected):\n            old_uid = str(old.SeriesInstanceUID) if old is not None else ""\n            new_uid = str(new.SeriesInstanceUID) if new is not None else ""\n            record = dict(StudyInstanceUID=str(uid), slot=slot, original_series_uid=old_uid,\n                          selected_series_uid=new_uid, changed=int(old_uid != new_uid))\n            for prefix, series in (("original", old), ("selected", new)):\n                for column in QUALITY_COLUMNS:\n                    record[f"{prefix}_{column}"] = series[column] if series is not None else None\n            records.append(record)\n    return pd.DataFrame(records)\n\n\ndef load_series(series_path, num_windows, image_size=DEFAULT_IMAGE_SIZE, crop_mm=DEFAULT_CROP_MM,\n                fluid_sensitive=0, fat_suppression=0, span_lo=0.02, span_hi=0.98):\n    """返回唯一切片缓存、窗口索引、原始位置及元数据；不跨序列构造窗口。"""\n    if num_windows < 1 or not 0 <= span_lo < span_hi <= 1:\n        raise ValueError("Invalid candidate count or span")\n    files, positions, headers = get_sorted_dicom_info(series_path)\n    first = int(len(files) * span_lo)\n    last = max(first, int(len(files) * span_hi) - 1)\n    # 覆盖 2%-98% 的原始索引范围；范围内继续采用 v1 物理位置采样。\n    anchors = first + _sample_by_position(positions[first:last + 1], num_windows)\n    neighbors = np.clip(anchors[:, None] + np.array([-1, 0, 1]), 0, len(files) - 1)\n    needed = np.unique(neighbors)\n    spacing = _get_pixel_spacing(headers, series_path)\n    volume = physical_center_crop(_read_sampled_slices(files, needed), spacing, crop_mm)\n    anchor_volume = volume[np.searchsorted(needed, anchors)]\n    foreground = anchor_volume[np.isfinite(anchor_volume) & (anchor_volume != 0)]\n    if not foreground.size:\n        raise ValueError(f"Series contains no finite non-zero anchor pixels after crop: {series_path}")\n    low, high = np.percentile(foreground, [0.5, 99.5])\n    volume = np.clip((volume - low) / (high - low + 1e-6), 0, 1).astype(np.float32)\n    slices = F.interpolate(torch.from_numpy(volume)[:, None], size=(image_size, image_size),\n                           mode="bilinear", align_corners=False)[:, 0]\n    window_indices = torch.from_numpy(np.searchsorted(needed, neighbors))\n    coverage = float(positions[-1] - positions[0])\n    anchor_positions = (2 * (positions[anchors] - positions[0]) / coverage - 1\n                        if coverage > 0 else np.zeros(num_windows))\n    header = headers[len(headers) // 2]\n    te, tr = float(getattr(header, "EchoTime", 0) or 0), float(getattr(header, "RepetitionTime", 0) or 0)\n    slice_spacing = float(np.median(np.abs(np.diff(positions)))) if len(positions) > 1 else 0\n    features = torch.tensor([\n        float(fluid_sensitive), float(fat_suppression), *_contrast_features(te, tr, int(fat_suppression)),\n        te / 100, tr / 5000, float(spacing.mean()), slice_spacing / 10, coverage / 200,\n    ], dtype=torch.float32)\n    return slices, window_indices, torch.as_tensor(anchor_positions, dtype=torch.float32), features\n\n\ndef sample_training_windows(slot_indices, count, rng):\n    """每个有效 slot 至少一个；剩余从未选候选随机抽取，候选不足时才重复。"""\n    slot_indices = np.asarray(slot_indices)\n    slots = np.unique(slot_indices)\n    if not len(slots) or count < len(slots):\n        raise ValueError("Training window count must cover every valid slot")\n    mandatory = np.array([rng.choice(np.flatnonzero(slot_indices == slot)) for slot in slots], dtype=np.int64)\n    remaining = np.setdiff1d(np.arange(len(slot_indices)), mandatory)\n    rng.shuffle(remaining)\n    take = min(count - len(mandatory), len(remaining))\n    chosen = np.concatenate([mandatory, remaining[:take]])\n    if len(chosen) < count:\n        chosen = np.concatenate([chosen, rng.choice(len(slot_indices), count - len(chosen), replace=True)])\n    return chosen\n\n\nclass KneeDataset(Dataset):\n    """v14 质量排序选序列；训练随机抽24，验证/推理取全部候选。"""\n    def __init__(self, study_df, series_df, image_size=DEFAULT_IMAGE_SIZE, crop_mm=DEFAULT_CROP_MM,\n                 train=False, train_windows=24, span_lo=0.02, span_hi=0.98, cache_dir=None, seed=42,\n                 series_selection=None):\n        if train_windows < len(SLOTS) or image_size < 1 or not 0 <= span_lo < span_hi <= 1:\n            raise ValueError("train_windows must be >=5; image_size/span must be valid")\n        self.study_df = study_df.reset_index(drop=True)\n        self.series_groups = {str(uid): group.reset_index(drop=True)\n                              for uid, group in series_df.groupby("StudyInstanceUID")}\n        self.image_size, self.crop_mm = image_size, crop_mm\n        self.train, self.train_windows, self.seed = train, train_windows, seed\n        validate_series_selection_config(series_selection)\n        self.series_selection = dict(series_selection, coverage_thresholds=dict(series_selection["coverage_thresholds"]))\n        self.span_lo, self.span_hi = span_lo, span_hi\n        self.cache_dir = Path(cache_dir) if cache_dir is not None else None\n        # persistent_workers 的 Dataset 副本共享 epoch，抽样不依赖 worker 数或访问顺序。\n        self.epoch = torch.zeros((), dtype=torch.int64).share_memory_()\n\n    def __len__(self):\n        return len(self.study_df)\n\n    def set_epoch(self, epoch):\n        self.epoch.fill_(epoch)\n\n    def _cache_path(self, uid, selected):\n        if self.cache_dir is None:\n            return None\n        sources = []\n        for record in selected:\n            if record is None:\n                sources.append(None)\n                continue\n            directory = Path(record.SeriesPath).resolve()\n            files = []\n            for file in sorted(directory.glob("*.dcm")):\n                stat = file.stat()\n                files.append((file.name, stat.st_size, stat.st_mtime_ns))\n            sources.append((str(directory), _binary_flag(record.Fluid_Sensitive),\n                            _binary_flag(record.Fat_Suppression), files))\n        signature = json.dumps([CACHE_SCHEMA, str(uid), self.image_size, self.crop_mm,\n                                self.span_lo, self.span_hi, SLOT_SPECS, sources], ensure_ascii=False)\n        return self.cache_dir / (hashlib.sha256(signature.encode("utf-8")).hexdigest() + ".pt")\n\n    def _build_bank(self, selected):\n        pixels, indices, positions, slots = [], [], [], []\n        metadata = torch.zeros(len(SLOTS), len(SERIES_FEATURES))\n        slot_mask = torch.zeros(len(SLOTS))\n        offset = 0\n        for slot, (record, (_, _, budget)) in enumerate(zip(selected, SLOT_SPECS)):\n            if record is None:\n                continue\n            slices, windows, depth, features = load_series(\n                record.SeriesPath, budget, self.image_size, self.crop_mm,\n                _binary_flag(record.Fluid_Sensitive) or 0, _binary_flag(record.Fat_Suppression) or 0,\n                self.span_lo, self.span_hi,\n            )\n            pixels.append(slices)\n            indices.append(windows + offset)\n            positions.append(depth)\n            slots.append(torch.full((budget,), slot, dtype=torch.long))\n            metadata[slot], slot_mask[slot] = features, 1\n            offset += len(slices)\n        if not pixels:\n            raise ValueError("Study has no usable series in the five slots")\n        return dict(cache_schema=CACHE_SCHEMA, slice_images=torch.cat(pixels),\n                    window_indices=torch.cat(indices), window_positions=torch.cat(positions),\n                    window_slot_indices=torch.cat(slots), series_features=metadata, slot_mask=slot_mask)\n\n    def candidate_bank(self, uid):\n        selected = select_slot_series(self.series_groups[str(uid)], self.series_selection, self.crop_mm, self.image_size)\n        path = self._cache_path(uid, selected)\n        if path is not None and path.is_file():\n            bank = torch.load(path, map_location="cpu", weights_only=True, mmap=True)\n            if bank.get("cache_schema") != CACHE_SCHEMA:\n                raise ValueError(f"Invalid input cache: {path}")\n            return bank\n        bank = self._build_bank(selected)\n        if path is not None:\n            path.parent.mkdir(parents=True, exist_ok=True)\n            temporary = path.with_name(f"{path.name}.{os.getpid()}.{uuid.uuid4().hex}.tmp")\n            try:\n                torch.save(bank, temporary)\n                try:\n                    os.replace(temporary, path)\n                except OSError:\n                    # 另一 DDP worker 已经写入并 mmap 时，Windows 可能拒绝覆盖。\n                    if not path.is_file():\n                        raise\n            finally:\n                temporary.unlink(missing_ok=True)\n        return bank\n\n    def __getitem__(self, index):\n        row = self.study_df.iloc[index]\n        uid = row.StudyInstanceUID\n        bank = self.candidate_bank(uid)\n        count = len(bank["window_indices"])\n        slots = bank["window_slot_indices"].numpy()\n        if self.train:\n            digest = hashlib.blake2b(f"{self.seed}|{int(self.epoch)}|{uid}".encode(), digest_size=8).digest()\n            rng = np.random.default_rng(int.from_bytes(digest, "little"))\n            chosen = sample_training_windows(slots, self.train_windows, rng)\n        else:\n            chosen = np.arange(count)\n        # 序列内按原始物理位置排序；不把随机抽取次序当作空间次序。\n        positions = bank["window_positions"].numpy()\n        chosen = chosen[np.lexsort((chosen, positions[chosen], slots[chosen]))]\n        chosen = torch.as_tensor(chosen, dtype=torch.long)\n        values = (pd.to_numeric(row[LABELS], errors="coerce").to_numpy(dtype=np.float32)\n                  if set(LABELS).issubset(row.index) else np.full(len(LABELS), np.nan, dtype=np.float32))\n        valid = ~np.isnan(values)\n        return dict(study_uid=str(uid), images=bank["slice_images"][bank["window_indices"][chosen]],\n                    window_positions=bank["window_positions"][chosen],\n                    window_slot_indices=bank["window_slot_indices"][chosen],\n                    slot_mask=bank["slot_mask"], series_features=bank["series_features"],\n                    candidate_count=torch.tensor(count),\n                    targets=torch.from_numpy(np.nan_to_num(values, nan=0)),\n                    label_mask=torch.from_numpy(valid.astype(np.float32)),\n                    label_weight=torch.from_numpy(np.where(valid, 2 * np.abs(values - 0.5), 0).astype(np.float32)))\n\n\ndef collate_studies(samples):\n    """只拼有效窗口；不同 Study 验证候选数不同时也不补齐图像。"""\n    batch = {key: torch.cat([sample[key] for sample in samples])\n             for key in ("images", "window_positions", "window_slot_indices")}\n    batch["window_batch_indices"] = torch.cat([\n        torch.full((len(sample["images"]),), i, dtype=torch.long) for i, sample in enumerate(samples)\n    ])\n    for key in ("slot_mask", "series_features", "targets", "label_mask", "label_weight", "candidate_count"):\n        batch[key] = torch.stack([sample[key] for sample in samples])\n    batch["study_uid"] = [sample["study_uid"] for sample in samples]\n    return batch\n\n\ndef build_datasets(root=DEFAULT_ROOT, **kwargs):\n    metadata = load_metadata(root)\n    train_df, train_series_df, test_df, test_series_df, sample_submission = metadata\n    quality_cache = Path(kwargs["cache_dir"]) / "series_quality" if kwargs.get("cache_dir") is not None else None\n    train_series_df = add_series_quality(train_series_df, quality_cache)\n    test_series_df = add_series_quality(test_series_df, quality_cache)\n    if kwargs.get("series_selection") is None:\n        weak_uids = train_df.loc[~train_df[LABELS].notna().all(axis=1), "StudyInstanceUID"]\n        kwargs["series_selection"] = make_series_selection_config(train_series_df[train_series_df.StudyInstanceUID.isin(weak_uids)])\n    metadata = (train_df, train_series_df, test_df, test_series_df, sample_submission)\n    return (KneeDataset(train_df, train_series_df, train=True, **kwargs),\n            KneeDataset(test_df, test_series_df, train=False, **kwargs), sample_submission, metadata)\n\n\n__all__ = ["DEFAULT_ROOT", "DEFAULT_IMAGE_SIZE", "DEFAULT_CROP_MM", "LABELS", "SLOTS", "SLOT_SPECS",\n           "CANDIDATE_BUDGETS", "load_metadata", "get_sorted_dicom_info", "read_dicom", "physical_center_crop",\n           "load_series", "select_slot_series", "sample_training_windows", "KneeDataset", "collate_studies",\n           "SERIES_SELECTION_VERSION", "QUALITY_COLUMNS", "read_series_quality", "add_series_quality",\n           "make_series_selection_config", "validate_series_selection_config", "series_selection_report"]\n', 'vlm_data.py': '"""Use v14 unchanged; cap evaluation windows before sending them to the VLM."""\n\nimport numpy as np\nimport torch\n\nfrom rsna_data import KneeDataset, SLOTS\n\n\ndef evaluation_indices(slots, positions, limit):\n    """Deterministic proportional slot allocation, evenly spaced within each slot."""\n    slots, positions = np.asarray(slots), np.asarray(positions)\n    if slots.shape != positions.shape or slots.ndim != 1 or not len(slots):\n        raise ValueError("Expected nonempty matching slot/position arrays")\n    if limit < 0 or (limit and limit < len(SLOTS)):\n        raise ValueError("eval_windows must be 0 (all) or >=5")\n    if not limit or limit >= len(slots):\n        return np.lexsort((np.arange(len(slots)), positions, slots))\n    active, counts = np.unique(slots, return_counts=True)\n    quotas = np.ones(len(active), dtype=np.int64)\n    desired = counts / counts.sum() * limit\n    while quotas.sum() < limit:\n        deficits = np.where(quotas < counts, desired - quotas, -np.inf)\n        quotas[np.argmax(deficits)] += 1\n    chosen = []\n    for slot, quota in zip(active, quotas):\n        indices = np.flatnonzero(slots == slot)\n        indices = indices[np.argsort(positions[indices], kind="stable")]\n        # Cell midpoints cover the whole series; no duplicate candidate indices.\n        offsets = np.floor((np.arange(quota) + 0.5) * len(indices) / quota).astype(int)\n        chosen.extend(indices[offsets])\n    chosen = np.asarray(chosen, dtype=np.int64)\n    return chosen[np.lexsort((chosen, positions[chosen], slots[chosen]))]\n\n\nclass VLMKneeDataset(KneeDataset):\n    def __init__(self, *args, eval_windows=0, **kwargs):\n        if eval_windows < 0 or (eval_windows and eval_windows < len(SLOTS)):\n            raise ValueError("eval_windows must be 0 or >=5")\n        super().__init__(*args, **kwargs)\n        self.eval_windows = eval_windows\n\n    def __getitem__(self, index):\n        sample = super().__getitem__(index)\n        if not self.train:\n            chosen = evaluation_indices(sample["window_slot_indices"].numpy(),\n                                        sample["window_positions"].numpy(), self.eval_windows)\n            chosen = torch.as_tensor(chosen, dtype=torch.long)\n            for key in ("images", "window_positions", "window_slot_indices"):\n                sample[key] = sample[key][chosen]\n        return sample\n', 'rsna_model.py': '"""Qwen3.5 study classifier: full vision tower + text-only LoRA + 12-logit head."""\n\nimport hashlib\nimport json\nfrom contextlib import contextmanager, nullcontext\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch import nn\n\nfrom rsna_data import LABELS, SLOTS\n\n\nARCHITECTURE = "baseline_v24_qwen35_2b_vision_full_text_lora"\nDEFAULT_QWEN_MODEL_DIR = Path(__file__).resolve().parents[1] / "Qwen3.5-2B"\nPROMPT_VERSION = "study_summary_v1"\nMODEL_INPUT_KEYS = {"input_ids", "attention_mask", "pixel_values", "image_grid_thw",\n                    "mm_token_type_ids", "position_ids"}\n\n\ndef load_processor(model_dir, image_size):\n    from transformers import AutoProcessor\n\n    processor = AutoProcessor.from_pretrained(str(model_dir), local_files_only=True)\n    processor.tokenizer.padding_side = "right"\n    patch = int(processor.image_processor.patch_size)\n    merge = int(processor.image_processor.merge_size)\n    if image_size < patch * merge or image_size % (patch * merge):\n        raise ValueError(f"image_size must be a positive multiple of {patch * merge}")\n    if processor.tokenizer.pad_token_id is None:\n        raise ValueError("Qwen tokenizer must define pad_token_id")\n    return processor\n\n\ndef processor_signature(processor):\n    """Guard tokenizer/template/normalization changes between training and inference."""\n    image = processor.image_processor\n    backend = json.loads(processor.tokenizer.backend_tokenizer.to_str())\n    # Padding/truncation are fixed by make_vlm_batch and may be mutated by calls.\n    backend.pop("padding", None)\n    backend.pop("truncation", None)\n    contract = {\n        "prompt_version": PROMPT_VERSION,\n        "chat_template": processor.chat_template,\n        "vocab": sorted(processor.tokenizer.get_vocab().items()),\n        "tokenizer_backend": backend,\n        "special_tokens": processor.tokenizer.special_tokens_map,\n        "pad_token_id": processor.tokenizer.pad_token_id,\n        "image_token_id": processor.image_token_id,\n        "patch_size": image.patch_size, "merge_size": image.merge_size,\n        "temporal_patch_size": image.temporal_patch_size,\n        "image_mean": image.image_mean, "image_std": image.image_std,\n        "do_resize": False, "do_rescale": False, "do_normalize": True,\n    }\n    payload = json.dumps(contract, sort_keys=True, ensure_ascii=False).encode("utf-8")\n    return hashlib.sha256(payload).hexdigest()\n\n\ndef make_vlm_batch(processor, batch, use_metadata=True, max_tokens=16384):\n    """Process v14 float [0,1] windows on CPU; retain processor-generated M-RoPE data."""\n    images = batch["images"]\n    if images.device.type != "cpu" or images.ndim != 4 or images.shape[1] != 3:\n        raise ValueError("Process packed [N,3,H,W] images on CPU before moving to GPU")\n    if not len(images) or not torch.isfinite(images).all() or images.min() < 0 or images.max() > 1:\n        raise ValueError("Expected finite nonempty v14 windows in [0,1]")\n    texts, ordered_images, image_counts = [], [], []\n    owners, slots, positions = (batch[key].numpy() for key in\n                               ("window_batch_indices", "window_slot_indices", "window_positions"))\n    size = len(batch["study_uid"])\n    if any(values.shape != (len(images),) for values in (owners, slots, positions)):\n        raise ValueError("Packed image maps differ in length")\n    if np.any((owners < 0) | (owners >= size)) or np.any((slots < 0) | (slots >= len(SLOTS))):\n        raise ValueError("Invalid study/slot image mapping")\n    for study in range(size):\n        content = [{"type": "text", "text":\n                    "These are ordered MRI windows from one knee study. "\n                    "The three image channels contain neighboring slices. "\n                    "Window position is normalized within each series.\\n"}]\n        count = 0\n        for slot, name in enumerate(SLOTS):\n            selected = np.flatnonzero((owners == study) & (slots == slot))\n            present = bool(batch["slot_mask"][study, slot])\n            if present != bool(len(selected)):\n                raise ValueError("Every present slot must have images, and absent slots must not")\n            if not present:\n                content.append({"type": "text", "text": f"\\n{name}: unavailable.\\n"})\n                continue\n            selected = selected[np.argsort(positions[selected], kind="stable")]\n            description = f"\\n{name}:"\n            if use_metadata:\n                flags = batch["series_features"][study, slot, :2].tolist()\n                description += f" fluid_sensitive={int(flags[0] > 0.5)}, fat_suppression={int(flags[1] > 0.5)};"\n            content.append({"type": "text", "text": description + "\\n"})\n            for index in selected:\n                content.append({"type": "text", "text": f"position={positions[index]:+.3f}\\n"})\n                content.append({"type": "image"})\n                ordered_images.append(images[index])\n                count += 1\n        if not count:\n            raise ValueError("Study contains no images")\n        content.append({"type": "text", "text":\n                        "\\nSummarize the evidence across all available series for these findings: "\n                        + ", ".join(LABELS) + "."})\n        texts.append(processor.apply_chat_template(\n            [{"role": "user", "content": content}], tokenize=False,\n            add_generation_prompt=True, enable_thinking=False))\n        image_counts.append(count)\n    # Inputs are already 384x384 and [0,1]. Apply only Qwen\'s normalization/patch packing.\n    encoded = processor(text=texts, images=ordered_images, padding=True, truncation=False,\n                        return_tensors="pt", do_resize=False, do_rescale=False, do_normalize=True)\n    grid = encoded["image_grid_thw"]\n    patch, merge = processor.image_processor.patch_size, processor.image_processor.merge_size\n    expected_grid = torch.tensor([1, images.shape[-2] // patch, images.shape[-1] // patch])\n    if grid.shape != (len(images), 3) or not torch.equal(grid.cpu(), expected_grid.expand_as(grid)):\n        raise ValueError("Processor changed image geometry; expected one grid per v14 window")\n    lengths = encoded["attention_mask"].sum(-1)\n    if lengths.max().item() > max_tokens:\n        raise ValueError(f"Prompt has {int(lengths.max())} tokens > max_tokens={max_tokens}; "\n                         "reduce windows/image size or explicitly raise the budget (no truncation)")\n    offset = 0\n    for study, count in enumerate(image_counts):\n        required = int((grid[offset:offset + count].prod(-1) // merge ** 2).sum())\n        actual = int((encoded["input_ids"][study] == processor.image_token_id).sum())\n        if actual != required:\n            raise ValueError(f"Image/token mismatch in study {study}: {actual} != {required}")\n        offset += count\n    result = {key: value for key, value in encoded.items() if key in MODEL_INPUT_KEYS}\n    for key in ("study_uid", "targets", "label_mask", "label_weight", "candidate_count"):\n        if key in batch:\n            result[key] = batch[key]\n    result["window_count"] = len(images)\n    return result\n\n\ndef _canonical_config(config):\n    def clean(value):\n        if isinstance(value, dict):\n            return {k: clean(v) for k, v in value.items()\n                    if k not in {"_name_or_path", "transformers_version", "architectures", "dtype", "torch_dtype",\n                                 "_attn_implementation_autoset"}}\n        if isinstance(value, list):\n            return [clean(v) for v in value]\n        return value\n    return clean(config.to_dict())\n\n\nclass RSNAQwen35(nn.Module):\n    def __init__(self, model_dir=DEFAULT_QWEN_MODEL_DIR, *, core=None, lora_rank=16, lora_alpha=32,\n                 lora_dropout=0.05, head_dropout=0.1, gradient_checkpointing=True,\n                 dtype=torch.bfloat16, attn_implementation="sdpa"):\n        super().__init__()\n        from peft import LoraConfig, get_peft_model\n        from transformers import Qwen3_5Model\n\n        if lora_rank < 1 or lora_alpha < 1 or not 0 <= lora_dropout < 1 or not 0 <= head_dropout < 1:\n            raise ValueError("Invalid LoRA/head configuration")\n        if core is None:\n            core, loading = Qwen3_5Model.from_pretrained(\n                str(model_dir), local_files_only=True, dtype=dtype,\n                attn_implementation=attn_implementation, output_loading_info=True)\n            if loading.get("missing_keys") or loading.get("mismatched_keys") or loading.get("error_msgs"):\n                raise ValueError(f"Incomplete/incompatible Qwen snapshot: {loading}")\n        self.vlm = core\n        self.base_config = _canonical_config(core.config)\n        self.model_settings = dict(lora_rank=lora_rank, lora_alpha=lora_alpha,\n                                   lora_dropout=lora_dropout, head_dropout=head_dropout)\n        core.requires_grad_(False)\n        # Discover exact text-layer Linear paths, including hybrid DeltaNet projections.\n        # Wrapping only language_model prevents accidental LoRA insertion into the vision tower.\n        targets = [name for name, module in core.language_model.named_modules()\n                   if name.startswith("layers.") and isinstance(module, nn.Linear)]\n        if not targets:\n            raise ValueError("No text-layer linear modules found for LoRA")\n        core.language_model = get_peft_model(core.language_model, LoraConfig(\n            r=lora_rank, lora_alpha=lora_alpha, lora_dropout=lora_dropout,\n            target_modules=targets, bias="none"))\n        self.lora_targets = targets\n        core.visual.requires_grad_(True)  # Includes the pretrained merger/projector.\n        core.visual.float()\n        for parameter in core.language_model.parameters():\n            if parameter.requires_grad:\n                parameter.data = parameter.data.float()\n        core.config.text_config.use_cache = False\n        if gradient_checkpointing:\n            core.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        self.head = nn.Sequential(nn.LayerNorm(core.config.text_config.hidden_size),\n                                  nn.Dropout(head_dropout),\n                                  nn.Linear(core.config.text_config.hidden_size, len(LABELS)))\n        self.assert_trainability()\n\n    def assert_trainability(self):\n        if not all(p.requires_grad for p in self.vlm.visual.parameters()):\n            raise RuntimeError("Entire vision tower/merger must be trainable")\n        active = [name for name, p in self.vlm.language_model.named_parameters() if p.requires_grad]\n        if not active or any("lora_" not in name for name in active):\n            raise RuntimeError("Only text LoRA parameters may be trainable")\n\n    def forward(self, input_ids, attention_mask, pixel_values, image_grid_thw, **kwargs):\n        inputs = dict(input_ids=input_ids, attention_mask=attention_mask,\n                      pixel_values=pixel_values, image_grid_thw=image_grid_thw)\n        inputs.update({key: value for key, value in kwargs.items() if key in MODEL_INPUT_KEYS})\n        # Use the base multimodal model, avoiding [B,T,248320] vocabulary logits.\n        output = self.vlm(**inputs, use_cache=False, return_dict=True)\n        locations = torch.arange(attention_mask.shape[1], device=attention_mask.device)\n        last = locations.expand_as(attention_mask).masked_fill(~attention_mask.bool(), -1).max(-1).values\n        if (last < 0).any():\n            raise ValueError("Empty attention mask")\n        pooled = output.last_hidden_state[torch.arange(len(last), device=last.device), last]\n        return self.head(pooled.float()).float()\n\n    def parameter_groups(self, vision_lr, lora_lr, head_lr, weight_decay):\n        categories = {"vision": [], "lora": [], "head": []}\n        for name, parameter in self.named_parameters():\n            if not parameter.requires_grad:\n                continue\n            category = "vision" if name.startswith("vlm.visual.") else "head" if name.startswith("head.") else "lora"\n            categories[category].append((name, parameter))\n        groups = []\n        for category, lr in (("vision", vision_lr), ("lora", lora_lr), ("head", head_lr)):\n            for decay in (True, False):\n                params = [p for _, p in categories[category] if (p.ndim >= 2) == decay]\n                if params:\n                    groups.append(dict(params=params, lr=lr, weight_decay=weight_decay if decay else 0,\n                                       name=f"{category}_{\'decay\' if decay else \'no_decay\'}"))\n        return groups\n\n    def trainable_state_dict(self):\n        return {name: p.detach().cpu().clone() for name, p in self.named_parameters() if p.requires_grad}\n\n    def load_trainable_state_dict(self, state):\n        expected = {name: p for name, p in self.named_parameters() if p.requires_grad}\n        if set(state) != set(expected):\n            missing, extra = set(expected) - set(state), set(state) - set(expected)\n            raise ValueError(f"Checkpoint trainable key mismatch: missing={sorted(missing)[:5]}, extra={sorted(extra)[:5]}")\n        for name, parameter in expected.items():\n            if state[name].shape != parameter.shape:\n                raise ValueError(f"Checkpoint shape mismatch: {name}")\n        with torch.no_grad():\n            for name, parameter in expected.items():\n                parameter.copy_(state[name].to(parameter))\n\n\ndef predict_batch(model, batch):\n    return model(**{key: value for key, value in batch.items() if key in MODEL_INPUT_KEYS})\n\n\n@contextmanager\ndef attention_context(model, implementation="sdpa", sdpa_backend="math"):\n    """Select evaluation kernels and restore both backbones for the next training epoch."""\n    from torch.nn.attention import SDPBackend, sdpa_kernel\n\n    raw = model.module if hasattr(model, "module") else model\n    core = raw.vlm\n    previous = {"": core.config._attn_implementation,\n                "text_config": core.config.text_config._attn_implementation,\n                "vision_config": core.config.vision_config._attn_implementation}\n    if sdpa_backend not in ("auto", "math"):\n        raise ValueError("sdpa_backend must be auto or math")\n    try:\n        core.set_attn_implementation(implementation)\n        for config in (core.config.text_config, core.config.vision_config):\n            if config._attn_implementation != implementation:\n                raise RuntimeError("Qwen attention backend did not switch; check Transformers version")\n        kernels = sdpa_kernel(SDPBackend.MATH) if implementation == "sdpa" and sdpa_backend == "math" else nullcontext()\n        with kernels:\n            yield\n    finally:\n        core.set_attn_implementation(previous)\n\n\ndef validate_checkpoint(checkpoint, model, signature):\n    from rsna_data import CANDIDATE_BUDGETS, validate_series_selection_config\n\n    if (checkpoint.get("architecture") != ARCHITECTURE or checkpoint.get("labels") != LABELS\n            or checkpoint.get("slots") != SLOTS or checkpoint.get("candidate_budgets") != CANDIDATE_BUDGETS):\n        raise ValueError("Requires a matching Baseline_v24 checkpoint")\n    validate_series_selection_config(checkpoint.get("series_selection"))\n    if checkpoint.get("model_settings") != model.model_settings or checkpoint.get("lora_targets") != model.lora_targets:\n        raise ValueError("LoRA/head configuration differs from checkpoint")\n    if checkpoint.get("base_config") != model.base_config:\n        raise ValueError("Qwen base configuration differs from checkpoint")\n    if checkpoint.get("processor_signature") != signature:\n        raise ValueError("Tokenizer/template/image normalization differs from training")\n\n\ndef load_classifier(checkpoint_path, model_dir, device, dtype=torch.bfloat16,\n                    attn_implementation="sdpa"):\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n    if checkpoint.get("architecture") != ARCHITECTURE:\n        raise ValueError("Not a Baseline_v24 checkpoint")\n    model = RSNAQwen35(model_dir, **checkpoint["model_settings"], gradient_checkpointing=False,\n                      dtype=dtype, attn_implementation=attn_implementation)\n    processor = load_processor(model_dir, checkpoint["args"]["image_size"])\n    validate_checkpoint(checkpoint, model, processor_signature(processor))\n    model.load_trainable_state_dict(checkpoint["trainable_model"])\n    for key in ("trainable_model", "training_model", "ema", "optimizer", "scheduler", "scaler", "rng_by_rank"):\n        checkpoint.pop(key, None)\n    return model.to(device).eval(), processor, checkpoint\n', 'inference.py': '"""Offline v24 inference, on one GPU or multiple GPUs via torchrun."""\n\nimport argparse\nfrom pathlib import Path\nimport os\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.distributed as dist\nfrom torch.utils.data import DataLoader\n\nfrom rsna_data import LABELS, _prepare_series_df, add_series_quality, collate_studies\nfrom rsna_model import DEFAULT_QWEN_MODEL_DIR, load_classifier, make_vlm_batch, predict_batch, attention_context\nfrom vlm_data import VLMKneeDataset\n\n\ndef submission_frame(test_df, predictions):\n    uids = test_df.StudyInstanceUID.astype(str).tolist()\n    if len(set(uids)) != len(uids) or set(uids) != set(predictions):\n        raise ValueError("Predicted UIDs must match test.csv exactly, without duplicates")\n    values = np.asarray([predictions[uid] for uid in uids], dtype=np.float32)\n    if not len(uids):\n        values = np.empty((0, len(LABELS)), dtype=np.float32)\n    if values.shape != (len(uids), len(LABELS)) or not np.isfinite(values).all():\n        raise ValueError("Predictions must be finite [num_studies,12]")\n    if np.any((values < 0) | (values > 1)):\n        raise ValueError("Predictions must be probabilities in [0,1]")\n    frame = pd.DataFrame(values, columns=LABELS)\n    frame.insert(0, "StudyInstanceUID", uids)\n    return frame\n\n\n@torch.inference_mode()\ndef run_inference(args):\n    torch.set_num_threads(args.cpu_threads)\n    world_size = int(os.environ.get("WORLD_SIZE", "1"))\n    rank, local_rank = int(os.environ.get("RANK", "0")), int(os.environ.get("LOCAL_RANK", "0"))\n    distributed = world_size > 1\n    if distributed and not torch.cuda.is_available():\n        raise RuntimeError("torchrun inference requires CUDA")\n    device = torch.device(f"cuda:{local_rank}" if torch.cuda.is_available() else "cpu")\n    if device.type == "cuda":\n        torch.cuda.set_device(device)\n    if distributed:\n        dist.init_process_group("nccl")\n    try:\n        amp = args.amp\n        if amp == "auto":\n            amp = "bf16" if device.type == "cuda" and torch.cuda.is_bf16_supported() else "fp16"\n        if device.type == "cpu":\n            amp = "none"\n        if amp == "bf16" and not torch.cuda.is_bf16_supported():\n            raise RuntimeError("Use --amp fp16 on a GPU without bf16 support")\n        dtype = {"bf16": torch.bfloat16, "fp16": torch.float16, "none": torch.float32}[amp]\n        model, processor, checkpoint = load_classifier(\n            args.checkpoint, args.qwen_model_dir, device, dtype=dtype,\n            attn_implementation=args.attn_implementation)\n        saved = checkpoint["args"]\n        test_df = pd.read_csv(args.data_root / "test.csv")\n        local_studies = test_df.iloc[rank::world_size].reset_index(drop=True)\n        series_df = pd.read_csv(args.data_root / "test_series.csv")\n        series_df = series_df[series_df.StudyInstanceUID.isin(local_studies.StudyInstanceUID)].copy()\n        series_df = _prepare_series_df(series_df, args.data_root / "test_series")\n        series_df = add_series_quality(series_df, cache_dir=None, workers=args.series_quality_workers)\n        eval_windows = saved["eval_windows"] if args.eval_windows is None else args.eval_windows\n        dataset = VLMKneeDataset(\n            local_studies, series_df, train=False, image_size=saved["image_size"], crop_mm=saved["crop_mm"],\n            train_windows=saved["train_windows"], eval_windows=eval_windows,\n            span_lo=saved["span_lo"], span_hi=saved["span_hi"], seed=saved["seed"],\n            series_selection=checkpoint["series_selection"], cache_dir=None)\n        loader = DataLoader(dataset, batch_size=args.batch_size, num_workers=args.num_workers,\n                            pin_memory=device.type == "cuda", collate_fn=collate_studies)\n        predictions = {}\n        for step, raw in enumerate(loader):\n            batch = make_vlm_batch(processor, raw, not saved["no_metadata"], saved["max_tokens"])\n            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}\n            with attention_context(model, args.attn_implementation, args.sdpa_backend):\n                with torch.autocast(device_type=device.type, dtype=dtype if amp != "none" else torch.bfloat16,\n                                    enabled=device.type == "cuda" and amp != "none"):\n                    probabilities = predict_batch(model, batch).sigmoid().cpu().numpy()\n            predictions.update(zip(batch["study_uid"], probabilities))\n            if (step + 1) % args.log_interval == 0 or step + 1 == len(loader):\n                print(f"rank={rank} studies={len(predictions)}/{len(local_studies)} "\n                      f"tokens={int(batch[\'attention_mask\'].sum(-1).max())}", flush=True)\n        gathered = [None] * world_size\n        if distributed:\n            dist.all_gather_object(gathered, predictions)\n        else:\n            gathered = [predictions]\n        if rank == 0:\n            merged = {}\n            for part in gathered:\n                if merged.keys() & part.keys():\n                    raise ValueError("Duplicate study prediction across ranks")\n                merged.update(part)\n            frame = submission_frame(test_df, merged)\n            args.output.parent.mkdir(parents=True, exist_ok=True)\n            frame.to_csv(args.output, index=False, lineterminator="\\n")\n            print(f"Saved {args.output}: {frame.shape}", flush=True)\n            return frame\n    finally:\n        if distributed and dist.is_initialized():\n            dist.destroy_process_group()\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--data-root", type=Path, required=True)\n    parser.add_argument("--checkpoint", type=Path, required=True)\n    parser.add_argument("--qwen-model-dir", type=Path, default=DEFAULT_QWEN_MODEL_DIR,\n                        help="Defaults to ../Qwen3.5-2B next to Baseline_v24")\n    parser.add_argument("--output", type=Path, default=Path("submission.csv"))\n    parser.add_argument("--batch-size", type=int, default=1)\n    parser.add_argument("--num-workers", type=int, default=2)\n    parser.add_argument("--cpu-threads", type=int, default=4)\n    parser.add_argument("--series-quality-workers", type=int, default=4)\n    parser.add_argument("--eval-windows", type=int, default=None)\n    parser.add_argument("--amp", choices=["auto", "bf16", "fp16", "none"], default="auto")\n    parser.add_argument("--attn-implementation", choices=["sdpa", "eager", "flash_attention_2"], default="sdpa")\n    parser.add_argument("--sdpa-backend", choices=["math", "auto"], default="math",\n                        help="Default math avoids fused attention kernels for full-window inference")\n    parser.add_argument("--log-interval", type=int, default=20)\n    args = parser.parse_args()\n    if min(args.batch_size, args.series_quality_workers, args.log_interval, args.cpu_threads) < 1 or args.num_workers < 0:\n        parser.error("Invalid batch/worker/log count")\n    if args.eval_windows is not None and args.eval_windows != 0 and args.eval_windows < 5:\n        parser.error("eval-windows must be 0 or >=5")\n    return args\n\n\nif __name__ == "__main__":\n    run_inference(parse_args())\n'}
CODE_DIR = Path("/kaggle/working/v24_code")
CODE_DIR.mkdir(exist_ok=True)
for name, source in SOURCES.items():
    with (CODE_DIR / name).open("w", encoding="utf-8", newline="\n") as handle:
        handle.write(source)
print("Exported shared v24 inference modules:", list(SOURCES))


In [ ]:
import subprocess
import sys

arguments = [str(CODE_DIR / "inference.py"), "--data-root", str(DATA_ROOT), "--checkpoint", str(CHECKPOINT_PATH),
             "--qwen-model-dir", str(QWEN_MODEL_DIR), "--output", str(OUTPUT),
             "--batch-size", str(BATCH_SIZE), "--num-workers", str(NUM_WORKERS),
             "--amp", AMP, "--attn-implementation", ATTENTION, "--sdpa-backend", SDPA_BACKEND]
if EVAL_WINDOWS is not None:
    arguments += ["--eval-windows", str(EVAL_WINDOWS)]
gpus = torch.cuda.device_count() if USE_ALL_GPUS else min(1, torch.cuda.device_count())
if gpus > 1:
    command = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={gpus}", *arguments]
else:
    command = [sys.executable, *arguments]
subprocess.check_call(command)


In [ ]:
import numpy as np
import pandas as pd

submission = pd.read_csv(OUTPUT)
test = pd.read_csv(DATA_ROOT / "test.csv")
expected_labels = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
assert submission.columns.tolist() == ["StudyInstanceUID", *expected_labels]
assert submission.StudyInstanceUID.tolist() == test.StudyInstanceUID.tolist()
values = submission[expected_labels].to_numpy()
assert np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()
print("submission.csv:", submission.shape)
display(submission.head())
